# Data Cleaning and Privacy Validation
Normalize whitespace and URLs without transliterating Telugu, remove exact duplicates only, and stop promotion if PII-like patterns require review.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_processing.cleaning import clean_frame
from src.data_processing.normalization import normalize_url
from src.data_processing.validation import SCHEMAS, scan_sensitive_text

In [ ]:
TEXT_COLUMNS = {'citizen_queries.csv': ['query'], 'complaint_queries.csv': ['query'], 'service_catalog.csv': ['service_name', 'description'], 'government_schemes.csv': ['scheme_name', 'description'], 'documents_and_forms.csv': ['document_name', 'description'], 'official_sources.csv': ['organization', 'department', 'service', 'description']}
URL_COLUMNS = {'official_sources.csv': ('url',), 'service_catalog.csv': ('source_url', 'official_application_url', 'official_form_url', 'status_tracking_url', 'complaint_url'), 'government_schemes.csv': ('application_url', 'source_url'), 'documents_and_forms.csv': ('download_url', 'source_url')}
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
cleaned_frames = {}
cleaning_summary = []
for filename, columns in SCHEMAS.items():
    frame = pd.read_csv(DATA_DIR / filename, keep_default_na=False)
    urls_to_check = [column for column in URL_COLUMNS.get(filename, ()) if column in frame]
    invalid_urls = sum(bool(str(value).strip()) and not normalize_url(value) for column in urls_to_check for value in frame[column])
    if invalid_urls:
        raise ValueError(f'{filename}: {invalid_urls} invalid URL(s) require review before cleaning')
    cleaned, summary = clean_frame(frame, URL_COLUMNS.get(filename, ()))
    text_columns = cleaned.select_dtypes(include=['object', 'category']).columns
    findings = scan_sensitive_text(cleaned, text_columns)
    if findings:
        raise ValueError(f'{filename}: review/redact sensitive-looking values before saving: {findings[:10]}')
    cleaned.to_csv(INTERIM_DIR / filename, index=False, encoding='utf-8-sig')
    cleaned_frames[filename] = cleaned
    cleaning_summary.append({'dataset': filename, 'invalid_urls': invalid_urls, **summary})
pd.DataFrame(cleaning_summary)

## Promote reviewed data
Processed outputs are not overwritten automatically. After reviewing the interim files and confirming their source provenance, change the flag below to `True` to promote the cleaned data.

In [ ]:
PROMOTE_REVIEWED_OUTPUTS = False
if PROMOTE_REVIEWED_OUTPUTS:
    for filename, frame in cleaned_frames.items():
        frame.to_csv(DATA_DIR / filename, index=False, encoding='utf-8-sig')
    print('Reviewed clean datasets promoted.')
else:
    print('Promotion skipped; reviewed intermediates remain in data/interim/.')